In [ ]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())

print("Click here to download the motion-dataset.zip: ", FileLink("motion-dataset.zip"))


inside dir:  ['requirements.txt', 'networks', 'dataset_processor.py', 'glove', 'exp_results', 'README.md', 'data_utils', 'checkpoints', '.git', 'options', 'main.ipynb', 'motion-dataset.zip', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils']
Click here to download the motion-dataset.zip:  /notebooks/motion-synthesis/motion-dataset.zip


In [2]:
import torch
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE
from networks.trainers import MotionVQVAETrainer

In [3]:
parser = TrainOptions()
options = parser.parse(args = ['--max_epoch', '10000'])
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.save_every_e = 100

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 196
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain

In [4]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_file = pjoin(options.data_root, 'train_micro.txt')
val_split_file = pjoin(options.data_root, 'val_micro.txt')

train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)
st = set()
for ds in train_dataset:
    st.add(ds['motion_parts'].shape)
print(st)
sample_motion = train_dataset[105]
print('Sample data shape: ', sample_motion['motion_parts'].shape)
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 8


100%|██████████| 8/8 [00:00<00:00, 146.88it/s]


Motion shape (B, T, D): (8, 199, 263)
Total number of motions 8, snippets 468
id list 4


100%|██████████| 4/4 [00:00<00:00, 167.93it/s]

Motion shape (B, T, D): (4, 170, 263)
Total number of motions 4, snippets 495
{(40, 6, 60)}
Sample data shape:  (40, 6, 60)


In [5]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=True, num_workers=1,
                              shuffle=False, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=options.batch_size, drop_last=True, num_workers=1,
                        shuffle=False, pin_memory=True)

vqvae = MotionVQVAE(
    input_dim=Dp_max,
    enc_hidden_dim=256,
    dec_hidden_dim=256,
    latent_dim=256,
    num_embeddings=256,
    beta=0.25
)

trainer = MotionVQVAETrainer(options, vqvae = vqvae)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader)


Number of epochs: 10000
Iters Per Epoch, Training: 0003, Validation: 003
Validation Loss: 0.58774 Reconstruction Loss: 0.46893 VQ Loss: 0.11881 Codebook Loss: 0.09505 Commitment Loss: 0.09505
epoch: 033 inner_iter:     0 1m 47s (- 533m 31s) niter: 0000100 completed:   0%) val_loss: 0.4654  loss: 0.9547  loss_rec: 0.8311  loss_vq: 0.1236  loss_codebook: 0.0989  loss_commit: 0.0989 
epoch: 066 inner_iter:     1 3m 32s (- 528m 5s) niter: 0000200 completed:   0%) val_loss: 0.4465  loss: 0.8485  loss_rec: 0.7897  loss_vq: 0.0588  loss_codebook: 0.0470  loss_commit: 0.0470 
epoch: 099 inner_iter:     2 5m 18s (- 526m 1s) niter: 0000300 completed:   1%) val_loss: 0.3955  loss: 0.8011  loss_rec: 0.7104  loss_vq: 0.0907  loss_codebook: 0.0725  loss_commit: 0.0725 
Validation Loss: 0.39119 Reconstruction Loss: 0.35533 VQ Loss: 0.03586 Codebook Loss: 0.02869 Commitment Loss: 0.02869
epoch: 133 inner_iter:     0 7m 5s (- 525m 20s) niter: 0000400 completed:   1%) val_loss: 0.3614  loss: 0.7103  los

: 